***

Preparing Workspace

***

In [ ]:


## Indicators:
# Cost_4


import pandas as pd
from pathlib import Path
from tqdm import tqdm
import plotly.express as px
from IPython.display import display
import sys
sys.path.append(str(Path.cwd().parent.parent / 'config'))
import functions as func


# SharePoint OneDrive paths
PATH_SERVER = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")
PATH_MAIN = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Data'
FILE_ABOUT = Path.home() / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents' / 'Process Revamp' / 'Task 6. Process Map' / 'About Indicators.xlsx'
# FILE_HOUSING = PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'SACOG Housing Dataset' /  'DOF_Summarized_00_25_detailed.xlsx' # Developed by Warren
FILE_HOUSING = PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'SACOG Housing Dataset' / 'Housing_Inventory_2026' /  'ya_08c_MnR_SACOG_Housing_Inventory_new_cols_HCD_20260601.xlsx' # Developed by Warren



def export_housing(df, params, path_out):   
    workbook_name = f'{params['indicator']} {params['geo']}_{params['source']}.xlsx'
    path_out_workbook = path_out / workbook_name
    if params['indicator'] in ['Cost_4']:
        if params['geo'] == 'MPO':
            params['geo'] = 'Six-County Sacramento Region'
    df_about = func.write_about(params)
    if params['indicator'] in ['Cost_4']:
        if params['geo'] == 'Six-County Sacramento Region':
            params['geo'] = 'MPO'
    with pd.ExcelWriter(path_out_workbook, mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
        df_about.to_excel(writer, sheet_name='About'      , index=False, header=False)
        df      .to_excel(writer, sheet_name=params['geo'], index=False,             )

        
EXPORT=False



In [ ]:


year_start = 2018
year_end   = 2025

years_to_import = range(year_start, year_end+1)

list_df = []

for year in years_to_import:
    df_year = pd.read_excel(FILE_HOUSING, sheet_name=str(year))
    df_year['Year'] = year
    list_df.append(df_year)

df_housing = pd.concat(list_df, ignore_index=True)
df_housing = df_housing[~df_housing['Jurisdiction'].str.contains('https')]
df_housing.loc[df_housing['Jurisdiction'].str.contains('COUNTY'), 'Jurisdiction'] = 'UNINCORPORATED'

df_housing.head()



In [ ]:


gb_jurisdiction = df_housing.groupby(['County', 'Jurisdiction', 'Year'], as_index=False)
gb_county       = df_housing.groupby(['County',                 'Year'], as_index=False)
gb_mpo          = df_housing.groupby([                          'Year'], as_index=False)

metrics = ['Total', 'INC_VLOW', 'INC_LOW', 'INC_MOD', 'INC_ABOVE_MOD']

#Jurisdiction level
print('Organizing indicator Cost_4 by Jurisdictions')
df_cost4_a = gb_jurisdiction[metrics].sum() 
display(df_cost4_a.head(5))

#County level 
print('Organizing indicator Cost_4 by Counties')
df_cost4_b = gb_county[metrics].sum()
display(df_cost4_b.head(5))

#MPO level 
print('Organizing indicator Cost_4 by MPO')
df_cost4_c = gb_mpo[metrics].sum()
display(df_cost4_c.head(5))


# Plotting
df_plot = df_cost4_c.copy()

df_plot = pd.melt(df_plot, id_vars = ['Year']) 
df_plot = df_plot[df_plot['variable'] != 'Total']
df_plot.columns = [col.lower() for col in df_plot.columns]
df_plot['percentage'] = 100*df_plot['value'] / df_plot.groupby(['year'])['value'].transform('sum')
df_plot = df_plot.sort_values(['year', 'variable'], ascending = [False, True])
display(df_plot.head())


fig = px.line(df_plot, x='year', y='percentage', color='variable', markers=True)
fig.update_layout(title = 'Housing Permit Green Zone Proportion SACOG')

fig.show()



In [ ]:


if EXPORT:

    indicator = 'Cost_4'
    source = 'SACOG HCD Summarized Data'
    sample_type = 'HCD'
    params = {
        'sample': sample_type
        , 'indicator': indicator
        , 'start_year': year_start
        , 'end_year': year_end
        , 'estimate': None
        , 'moe_thresh': None
        , 'source': source
        , 'geo': 'Six-County Sacramento Region'
    }

    # Update overall about documentations workbook
    df_about = func.write_about(params)
    with pd.ExcelWriter(FILE_ABOUT, mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace') as writer:
        df_about.to_excel(writer, index = False, sheet_name = indicator, header = False)


    print('Exporting...')
    print()
    paths_out = [PATH_MAIN / 'Vibrant and Inclusive Places' / 'Development' / 'Housing Cost' / f'{indicator} RHNA Income', PATH_SERVER]

    for path_out in paths_out:
        
        params['geo'] = 'Jurisdictions'
        export_housing(df_cost4_a, params, path_out)
            
        params['geo'] = 'Counties'
        export_housing(df_cost4_b, params, path_out)
        
        params['geo'] = 'MPO'
        export_housing(df_cost4_c, params, path_out)

    print('Success!!')

    